# Enrichissement FAOSTAT — Production 2014-2023

---
## 1. Installation & imports

In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import time
from pathlib import Path

# Chemins
DATA_RAW  = Path('../data/raw')
CACHE_DIR = DATA_RAW / 'faostat_cache'
CACHE_DIR.mkdir(exist_ok=True)

FAO_HISTORIQUE = DATA_RAW / 'FAO.csv'
OUTPUT_FILE    = DATA_RAW / 'faostat_2014_2023.csv'
OUTPUT_COMPLET = DATA_RAW / 'FAO_complet_1961_2023.csv'

# URL de bulk download FAOSTAT
# Food Balance Sheets (données récentes, post-2010) — même structure que FAO.csv
URL_FBS = "https://bulks-faostat.fao.org/production/FoodBalanceSheets_E_All_Data.zip"

print("✅ Imports OK")
print(f"📂 Cache : {CACHE_DIR.absolute()}")
print(f"🌐 Source : {URL_FBS}")

In [ ]:
import faostat
import pandas as pd
import numpy as np
import time
from pathlib import Path
from importlib.metadata import version

# Chemins
DATA_RAW = Path('../data/raw')
CACHE_DIR = DATA_RAW / 'faostat_cache'
CACHE_DIR.mkdir(exist_ok=True)

FAO_HISTORIQUE = DATA_RAW / 'FAO.csv'
OUTPUT_FILE    = DATA_RAW / 'faostat_2014_2023.csv'
OUTPUT_COMPLET = DATA_RAW / 'FAO_complet_1961_2023.csv'

print(f"✅ faostat version : {version('faostat')}")
print(f"📂 Cache : {CACHE_DIR.absolute()}")

In [ ]:
# 2.1 — Téléchargement du fichier ZIP FAOSTAT
# Le cache évite de re-télécharger si le fichier existe déjà

CACHE_ZIP = CACHE_DIR / 'FoodBalanceSheets_E_All_Data.zip'
CACHE_CSV = CACHE_DIR / 'FoodBalanceSheets_E_All_Data.csv'

if CACHE_CSV.exists():
    print(f"📂 Fichier déjà en cache : {CACHE_CSV.name}")
    print(f"   Taille : {CACHE_CSV.stat().st_size / 1024**2:.1f} MB")
else:
    print(f"⏳ Téléchargement en cours...")
    print(f"   URL : {URL_FBS}")
    print(f"   (peut prendre 1-3 minutes selon la connexion)\n")

    response = requests.get(URL_FBS, stream=True, timeout=120)
    response.raise_for_status()

    # Afficher la progression
    total = int(response.headers.get('content-length', 0))
    downloaded = 0
    chunks = []
    for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
        chunks.append(chunk)
        downloaded += len(chunk)
        if total:
            pct = downloaded / total * 100
            print(f"   {pct:.0f}% — {downloaded / 1024**2:.1f} / {total / 1024**2:.1f} MB", end='\r')

    print(f"\n✅ Téléchargé : {downloaded / 1024**2:.1f} MB")

    # Extraire le CSV du ZIP
    print("📦 Extraction du ZIP...")
    zip_bytes = b''.join(chunks)
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        print(f"   Fichiers dans le ZIP : {z.namelist()}")
        csv_name = [f for f in z.namelist() if f.endswith('.csv') and 'All_Data' in f][0]
        with z.open(csv_name) as f:
            content = f.read()

    # Sauvegarder en cache
    CACHE_CSV.write_bytes(content)
    print(f"✅ CSV extrait et mis en cache : {CACHE_CSV.name}")
    print(f"   Taille : {CACHE_CSV.stat().st_size / 1024**2:.1f} MB")

In [ ]:
# 2.2 — Chargement et inspection du fichier téléchargé
print("⏳ Chargement du CSV...")
df_fbs = pd.read_csv(CACHE_CSV, encoding='latin-1', low_memory=False)

print(f"✅ Chargé : {len(df_fbs):,} lignes × {len(df_fbs.columns)} colonnes")
print(f"\nColonnes : {df_fbs.columns.tolist()}")

# Colonnes années disponibles
year_cols_fbs = [c for c in df_fbs.columns if c.startswith('Y') and c[1:].isdigit()]
years_in_file = sorted([int(c[1:]) for c in year_cols_fbs])
print(f"\nPériode couverte : {years_in_file[0]}–{years_in_file[-1]}")
print(f"Années disponibles : {len(year_cols_fbs)}")

print(f"\nÉléments disponibles :")
print(df_fbs[['Element Code', 'Element']].drop_duplicates().sort_values('Element Code').to_string(index=False))

df_fbs.head()

In [ ]:
# 2.3 — Filtrer uniquement les années et éléments qui correspondent à FAO.csv
print("🔍 Éléments dans FAO.csv :")
df_historique = pd.read_csv(FAO_HISTORIQUE, encoding='latin-1')
elements_fao = df_historique[['Element Code', 'Element']].drop_duplicates().sort_values('Element Code')
print(elements_fao.to_string(index=False))

# Filtrer les éléments qui correspondent
ELEMENT_CODES = elements_fao['Element Code'].tolist()
elements_communs = df_fbs[df_fbs['Element Code'].isin(ELEMENT_CODES)][['Element Code', 'Element']].drop_duplicates()
print(f"\n✅ Éléments communs FAO.csv ↔ FBS : {len(elements_communs)}")
print(elements_communs.to_string(index=False))

# Années à extraire (ce qui manque après 2013)
ANNEES_CIBLES = [c for c in year_cols_fbs if int(c[1:]) >= 2014]
print(f"\n🎯 Colonnes à extraire : {ANNEES_CIBLES}")

In [ ]:
# 2.4 — Diagnostic de l'écart et identification des agrégats régionaux
print("🔍 Diagnostic de l'écart FAO.csv ↔ FBS...\n")

# Comparer les listes de pays
pays_fao = set(df_historique['Area'].unique())
pays_fbs = set(df_fbs['Area'].unique())

pays_fbs_seulement = pays_fbs - pays_fao
pays_fao_seulement = pays_fao - pays_fbs

print(f"Pays dans FAO.csv           : {len(pays_fao)}")
print(f"Pays dans FBS               : {len(pays_fbs)}")
print(f"Pays communs                : {len(pays_fao & pays_fbs)}")
print(f"Pays FBS non présents dans FAO.csv (agrégats probables) : {len(pays_fbs_seulement)}")

print(f"\n📋 Exemples d'agrégats dans FBS (non présents dans FAO.csv) :")
for p in sorted(pays_fbs_seulement)[:30]:
    print(f"   - {p}")

# Vérifier si ce sont bien des agrégats en comparant les totaux
# FAO.csv : pays individuels uniquement
# FBS     : pays individuels + agrégats → double comptage → écart x4-5
print(f"\n💡 Conclusion : le fichier FBS inclut des agrégats régionaux qui gonflent le total.")
print(f"   Solution : filtrer en gardant uniquement les pays présents dans FAO.csv.")

# Définir la liste des pays à garder (intersection)
PAYS_A_GARDER = pays_fao & pays_fbs
print(f"\n✅ Pays à conserver : {len(PAYS_A_GARDER)}")
print(f"⚠️  Pays FAO.csv absents de FBS (à surveiller) : {len(pays_fao_seulement)}")
if pays_fao_seulement:
    for p in sorted(pays_fao_seulement)[:10]:
        print(f"   - {p}")

In [ ]:
# -- Section 3 — Extraction des années 2014-2023 (pays individuels uniquement) --

print("=" * 60)
print("📦 SECTION 3 — Extraction des années 2014-2023")
print("=" * 60)

# Colonnes à garder : identifiants + années cibles
id_cols       = [c for c in df_fbs.columns if not (c.startswith('Y') and c[1:].isdigit())]
cols_a_garder = id_cols + ANNEES_CIBLES

# Double filtre : éléments Food/Feed + pays individuels uniquement (sans agrégats)
df_recent = (
    df_fbs[
        df_fbs['Element Code'].isin(ELEMENT_CODES) &
        df_fbs['Area'].isin(PAYS_A_GARDER)
    ][cols_a_garder]
    .copy()
)

print(f"\n✅ Extraction terminée :")
print(f"   Lignes    : {len(df_recent):,}")
print(f"   Pays      : {df_recent['Area'].nunique()}")
print(f"   Produits  : {df_recent['Item'].nunique()}")
print(f"   Éléments  : {df_recent['Element'].unique().tolist()}")
print(f"   Années    : {[c[1:] for c in ANNEES_CIBLES]}")

# Vérification cohérence sur 2013 APRÈS filtre
total_fbs_filtre = df_fbs[
    (df_fbs['Element Code'] == ELEMENT_CODES[0]) &
    (df_fbs['Area'].isin(PAYS_A_GARDER))
]['Y2013'].sum()
total_fao_2013 = df_historique[df_historique['Element Code'] == ELEMENT_CODES[0]]['Y2013'].sum()

diff_pct = abs(total_fbs_filtre - total_fao_2013) / total_fao_2013 * 100 if total_fao_2013 else 0
print(f"\n🔄 Cohérence 2013 après filtre pays :")
print(f"   FAO.csv total : {total_fao_2013:,.0f}")
print(f"   FBS filtré    : {total_fbs_filtre:,.0f}")
print(f"   Écart         : {diff_pct:.1f}%")
if diff_pct < 10:
    print("   ✅ Cohérence validée")
else:
    print("   ⚠️  Écart encore important — certains pays ont changé de nom entre les deux fichiers")

df_recent.head()

---
## 4. Concaténation

On assemble les données de l'API avec FAO.csv historique.

In [ ]:
# 4.1 — df_recent est déjà disponible depuis le bulk download (section 3)
# On le transforme du format wide (colonnes Y2014...) au format long (une ligne par année)
print("🔄 Transformation df_recent (wide → long)...")

id_cols_recent = [c for c in df_recent.columns if not (c.startswith('Y') and c[1:].isdigit())]

df_recent_long = df_recent.melt(
    id_vars   = id_cols_recent,
    value_vars = ANNEES_CIBLES,
    var_name  = 'Year_label',
    value_name= 'Value'
)
df_recent_long['Year'] = df_recent_long['Year_label'].str.replace('Y', '').astype(int)
df_recent_long = df_recent_long.drop(columns=['Year_label'])

print(f"  Lignes : {len(df_recent_long):,}")
print(f"  Années : {sorted(df_recent_long['Year'].unique())}")
df_recent_long.head()

In [ ]:
# 4.2 — Transformation FAO.csv historique (wide → long)
print("🔄 Transformation FAO.csv historique (wide → long)...")

year_cols_hist = [c for c in df_historique.columns if c.startswith('Y') and c[1:].isdigit()]
id_cols_hist   = [c for c in df_historique.columns if c not in year_cols_hist]

df_hist_long = df_historique.melt(
    id_vars   = id_cols_hist,
    value_vars = year_cols_hist,
    var_name  = 'Year_label',
    value_name= 'Value'
)
df_hist_long['Year'] = df_hist_long['Year_label'].str.replace('Y', '').astype(int)
df_hist_long = df_hist_long.drop(columns=['Year_label'])

print(f"  Lignes : {len(df_hist_long):,}")
print(f"  Années : {df_hist_long['Year'].min()}–{df_hist_long['Year'].max()}")
df_hist_long.head()

In [ ]:
# 4.3 — Aligner les colonnes et concaténer
print("📐 Colonnes FAO historique :", sorted(df_hist_long.columns.tolist()))
print("📐 Colonnes FBS récent     :", sorted(df_recent_long.columns.tolist()))

# Colonnes communes à conserver dans les deux sources
COLS_FINALES = ['Area Code', 'Area', 'Item Code', 'Item', 'Element Code', 'Element', 'Unit', 'Year', 'Value']

cols_hist   = [c for c in COLS_FINALES if c in df_hist_long.columns]
cols_recent = [c for c in COLS_FINALES if c in df_recent_long.columns]

df_hist_final   = df_hist_long[cols_hist].copy()
df_recent_final = df_recent_long[cols_recent].copy()

df_hist_final['Source']   = 'FAO_historique'
df_recent_final['Source'] = 'FBS_bulk'

print(f"\n  Colonnes retenues hist   : {cols_hist}")
print(f"  Colonnes retenues recent : {cols_recent}")

In [ ]:
# 4.4 — Concaténation et nettoyage
print("🔗 Concaténation finale...")
df_complet = pd.concat([df_hist_final, df_recent_final], ignore_index=True)

avant = len(df_complet)
df_complet = df_complet.dropna(subset=['Value'])
apres = len(df_complet)

print(f"\n📊 Résultat :")
print(f"  FAO historique  : {len(df_hist_final):,} lignes  ({df_hist_final['Year'].min()}–{df_hist_final['Year'].max()})")
print(f"  FBS récent      : {len(df_recent_final):,} lignes ({df_recent_final['Year'].min()}–{df_recent_final['Year'].max()})")
print(f"  Total avant     : {avant:,}")
print(f"  Nulls supprimés : {avant - apres:,}")
print(f"  Total final     : {apres:,}")
print(f"  Pays            : {df_complet['Area'].nunique()}")
print(f"  Produits        : {df_complet['Item'].nunique()}")
print(f"  Période         : {df_complet['Year'].min()}–{df_complet['Year'].max()}")

df_complet.sample(5)

In [ ]:
# 4.5 — Sauvegarde
print("💾 Sauvegarde...")

df_recent_final.to_csv(OUTPUT_FILE, index=False)
print(f"  ✅ {OUTPUT_FILE.name} ({len(df_recent_final):,} lignes) — données 2014-2023")

df_complet.to_csv(OUTPUT_COMPLET, index=False)
print(f"  ✅ {OUTPUT_COMPLET.name} ({len(df_complet):,} lignes) — données complètes 1961-2023")

---
## 5. Validation

In [ ]:
# 5.1 — Couverture temporelle : lignes par année
print("📅 Lignes par année :")
lignes_par_annee = df_complet.groupby('Year').size().reset_index(name='nb_lignes')

# Repérer visuellement la jonction 2013/2014
print("\n  (autour de la jonction FAO ↔ API)")
jonction = lignes_par_annee[
    lignes_par_annee['Year'].between(2011, 2016)
]
print(jonction.to_string(index=False))

# Vérifier qu'il n'y a pas de chute brutale à 2014
nb_2013 = lignes_par_annee.loc[lignes_par_annee['Year'] == 2013, 'nb_lignes'].values
nb_2014 = lignes_par_annee.loc[lignes_par_annee['Year'] == 2014, 'nb_lignes'].values

if len(nb_2013) and len(nb_2014):
    ecart = abs(nb_2013[0] - nb_2014[0]) / nb_2013[0] * 100
    print(f"\n  Écart 2013↔2014 : {ecart:.1f}%")
    if ecart < 20:
        print("  ✅ Jonction cohérente")
    else:
        print("  ⚠️  Écart important — vérifier la couverture pays/produits entre les deux sources")

In [ ]:
# 5.2 — Vérifier les doublons (un pays-produit-année-élément ne doit apparaître qu'une fois)
print("🔄 Vérification des doublons...")

cles = ['Area', 'Item', 'Element', 'Year']
doublons = df_complet.duplicated(subset=cles).sum()
print(f"  Doublons : {doublons:,}")

if doublons > 0:
    print("  ⚠️  Doublons détectés — vérifier le chevauchement 2013")
    print(df_complet[df_complet.duplicated(subset=cles, keep=False)].head(10))
else:
    print("  ✅ Aucun doublon")

In [ ]:
# 5.2b — Déduplication
# Les doublons viennent de FAO.csv : certains produits ont deux Item Codes différents
# pour le même Item name (ex: "Eggs" → codes 2744 et 2949, deux classifications FAO).
# On garde la première occurrence par (Area, Item, Element, Year).

if doublons > 0:
    print("🔧 Diagnostic des doublons...")
    cles = ['Area', 'Item', 'Element', 'Year']
    df_doublons = df_complet[df_complet.duplicated(subset=cles, keep=False)]

    print(f"  Lignes impliquées : {len(df_doublons):,}")
    print(f"  Sources concernées :\n{df_doublons['Source'].value_counts().to_string()}")
    print(f"\n  Item Codes distincts pour les doublons :")
    print(df_doublons.groupby('Item')['Item Code'].nunique().sort_values(ascending=False).head(10).to_string())

    print(f"\n  Exemple :")
    ex = df_doublons[df_doublons['Item'] == df_doublons['Item'].iloc[0]].head(4)
    print(ex[['Area', 'Item', 'Item Code', 'Element', 'Year', 'Value', 'Source']].to_string(index=False))

    # Déduplication : on garde la première occurrence par clé métier
    avant = len(df_complet)
    df_complet = df_complet.drop_duplicates(subset=cles, keep='first').reset_index(drop=True)
    apres = len(df_complet)

    print(f"\n✅ Déduplication :")
    print(f"  Avant  : {avant:,} lignes")
    print(f"  Après  : {apres:,} lignes")
    print(f"  Supprimé : {avant - apres:,} doublons")

    # Vérification
    restants = df_complet.duplicated(subset=cles).sum()
    print(f"  Doublons restants : {restants} {'✅' if restants == 0 else '⚠️'}")

    # Re-sauvegarder avec les données dédupliquées
    print("\n💾 Re-sauvegarde des fichiers sans doublons...")
    df_complet.to_csv(OUTPUT_COMPLET, index=False)
    print(f"  ✅ {OUTPUT_COMPLET.name} mis à jour ({apres:,} lignes)")
else:
    print("  ✅ Aucun doublon — pas de traitement nécessaire")

In [ ]:
# 5.3 — Vérification production mondiale blé (traceur fiable)
print("🌾 Évolution production mondiale blé (traceur) :")

ble_keywords = ['Wheat', 'wheat']
masque_ble   = df_complet['Item'].str.contains('|'.join(ble_keywords), case=False, na=False)

ble_annuel = (
    df_complet[masque_ble]
    .groupby('Year')['Value']
    .sum()
    .reset_index()
)

# Afficher autour de la jonction
print(ble_annuel[
    ble_annuel['Year'].between(2010, 2023)
].to_string(index=False))

print("\n✅ La tendance doit être progressive (pas de saut brutal entre 2013 et 2014)")

In [ ]:
# 5.4 — Résumé final
print("=" * 60)
print("✅ ENRICHISSEMENT FAOSTAT TERMINÉ")
print("=" * 60)
print(f"""
Fichiers produits :
  📄 {OUTPUT_FILE.name}
     → Données API 2014-2023 uniquement
     → {len(df_recent_final):,} lignes

  📄 {OUTPUT_COMPLET.name}
     → Données complètes 1961-2023
     → {len(df_complet):,} lignes
     → {df_complet['Area'].nunique()} pays
     → {df_complet['Item'].nunique()} produits
     → {df_complet['Year'].min()}–{df_complet['Year'].max()}""")